<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/churn_agglomerative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/Churn_Modelling.csv")


# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Exited"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Exited"]
)

# Verificando os tamanhos
print(f"Treino: {len(df_treino)} ({len(df_treino)/len(df):.1%})")
print(f"Teste: {len(df_teste)} ({len(df_teste)/len(df):.1%})")
print(f"Validação: {len(df_validacao)} ({len(df_validacao)/len(df):.1%})")

# Verificando a distribuição de Exited
print("\nDistribuição no DF original:")
print(df["Exited"].value_counts(normalize=True))

print("\nDistribuição no treino:")
print(df_treino["Exited"].value_counts(normalize=True))

print("\nDistribuição no teste:")
print(df_teste["Exited"].value_counts(normalize=True))

print("\nDistribuição na validação:")
print(df_validacao["Exited"].value_counts(normalize=True))

Treino: 7000 (70.0%)
Teste: 1500 (15.0%)
Validação: 1500 (15.0%)

Distribuição no DF original:
Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

Distribuição no treino:
Exited
0    0.796286
1    0.203714
Name: proportion, dtype: float64

Distribuição no teste:
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64

Distribuição na validação:
Exited
0    0.796667
1    0.203333
Name: proportion, dtype: float64


In [5]:
df_treino = df_treino.rename(columns={
    'RowNumber': 'NumeroLinha',
    'CustomerId': 'IdCliente',
    'Surname': 'Sobrenome',
    'CreditScore': 'PontuacaoCredito',
    'Geography': 'Geografia',
    'Gender': 'Genero',
    'Age': 'Idade',
    'Tenure': 'TempoRelacionamento',
    'Balance': 'Saldo',
    'NumOfProducts': 'NumeroProdutos',
    'HasCrCard': 'PossuiCartaoCredito',
    'IsActiveMember': 'MembroAtivo',
    'EstimatedSalary': 'SalarioEstimado',
    'Exited': 'Saiu'
})

In [6]:
from sklearn.preprocessing import StandardScaler

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

df_treino_cancelamento = df_treino[df_treino["Saiu"] == 1][variaveis_numericas]

# Scaler específico para a base de cancelados (não reaproveitar o scaler do df_treino completo)
scaler_cancelamento = StandardScaler()
X_scaled = scaler_cancelamento.fit_transform(df_treino_cancelamento)

In [7]:
def calcular_estabilidade_agglomerative(X, k, linkage, labels_referencia,
                                          n_bootstrap=30, sample_frac=0.8, random_state=42):
    rng = np.random.default_rng(random_state)
    X_arr = np.asarray(X)
    n_samples = X_arr.shape[0]
    ari_scores = []

    for i in range(n_bootstrap):
        seed = int(rng.integers(0, 1_000_000))
        idx_sample = resample(
            np.arange(n_samples), replace=True,
            n_samples=int(n_samples * sample_frac), random_state=seed
        )
        idx_sample = np.unique(idx_sample)
        X_sample = X_arr[idx_sample]

        modelo_boot = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels_boot = modelo_boot.fit_predict(X_sample)

        labels_ref_sample = np.asarray(labels_referencia)[idx_sample]
        ari_scores.append(adjusted_rand_score(labels_ref_sample, labels_boot))

    return np.mean(ari_scores), np.std(ari_scores)


resultados_agglomerative = []
linkage_usado = 'ward'

# X_scaled aqui é o mesmo gerado a partir do df_treino_cancelamento
for k in range(2, 8):
    modelo = AgglomerativeClustering(n_clusters=k, linkage=linkage_usado)
    labels = modelo.fit_predict(X_scaled)

    estab_media, estab_std = calcular_estabilidade_agglomerative(
        X_scaled, k, linkage_usado, labels
    )

    resultados_agglomerative.append({
        'k': k,
        'linkage': linkage_usado,
        'silhouette': silhouette_score(X_scaled, labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(X_scaled, labels),
        'estabilidade_media_ari': estab_media,
        'estabilidade_std_ari': estab_std,
    })

df_resultados_agglomerative = pd.DataFrame(resultados_agglomerative).set_index('k')
df_resultados_agglomerative

,linkage,silhouette,davies_bouldin,calinski_harabasz,estabilidade_media_ari,estabilidade_std_ari
k,,,,,,
2,ward,0.163820,2.184053,191.807458,0.159818,0.237199
3,ward,0.161602,1.998816,212.370945,0.657360,0.130246
4,ward,0.118939,2.030779,192.903710,0.422676,0.079718
5,ward,0.101755,1.830177,177.065844,0.438498,0.068698
6,ward,0.090884,1.726819,160.201578,0.443054,0.064806
7,ward,0.096731,1.847253,149.763767,0.379738,0.057584
